# MMA-DFER + UOT — Kaggle pipelineMục đích: **kiểm chứng pipeline chạy đúng**, không phải lấy số cuối cùng(Kaggle giới hạn ~12h/phiên nên không đủ cho 5 fold × 25 epoch).Chạy tuần tự. Chỉ cần sửa **Cell 2 (CONFIG)** — các cell sau tự dùng lại biến ở đó.Tài liệu: `KAGGLE.md`, `SETUP.md`, `UOT_INTEGRATION.md`, `UOT_CODE_WALKTHROUGH.md`.

## 1. Clone repo

In [ ]:
REPO   = "https://github.com/YouttyLe-DSAI/DEFR-UOT.git"BRANCH = "feat/uot-fusion"WORK   = "/kaggle/working/DEFR-UOT"import os, shutilif os.path.exists(WORK):    shutil.rmtree(WORK)          # rerun-safe!git clone -b {BRANCH} --depth 1 {REPO} {WORK}%cd {WORK}!git log --oneline -1# Repo private? Lưu token trong Add-ons -> Secrets (tên: GH_TOKEN), rồi dùng:# from kaggle_secrets import UserSecretsClient# tok = UserSecretsClient().get_secret("GH_TOKEN")# !git clone -b {BRANCH} https://{tok}@github.com/YouttyLe-DSAI/DEFR-UOT.git {WORK}

## 2. CONFIG — cell duy nhất cần sửaMặc định `FRAMES`/`AUDIO` để trống → **Cell 5 tự dò theo nội dung**, không cần biếtslug Kaggle. Tên hiển thị trong sidebar là *tiêu đề* dataset, còn đường dẫn mount dùng*slug* — hai thứ này thường khác nhau, nên đừng chép tay từ sidebar.Chỉ điền tay khi auto dò sai/thiếu, và lấy đường dẫn thật từ output Cell 4.

In [ ]:
DATASET  = "MAFW"                          # "MAFW" hoặc "DFEW"DATA     = "/kaggle/temp/data"             # cây symlink; /kaggle/temp không tính vào quota 20GBCKPT_DIR = "/kaggle/input/mma-dfer-pretrained"   # chứa 2 file .pth, xem Cell 7# Tham số train — chọn cho GPU Kaggle (T4 16GB, 2-4 vCPU)EPOCHS, BATCH, LR, WORKERS, FOLD = 5, 4, 7e-5, 2, 1# Để trống = tự dò. Chỉ điền khi Cell 5 báo dò thiếu.FRAMES = []AUDIO  = []ANN = (f"annotation/MAFW_set_{FOLD}_train_faces.txt" if DATASET == "MAFW"       else f"annotation/DFEW_set_{FOLD}_train.txt")ANN_TEST    = ANN.replace("train", "test")FRAMES_ROOT = f"{DATA}/mfaw/clips_faces" if DATASET == "MAFW" else f"{DATA}/dfew/clip_224x224"SRC_ARG     = ("--auto" if not FRAMES               else f'--frames {" ".join(FRAMES)} --audio {" ".join(AUDIO)}')print(DATASET, "| nguồn:", "tự dò" if not FRAMES else "chỉ định tay")print("frames root sẽ dựng tại:", FRAMES_ROOT)

## 3. Dependencies`timm==0.9.16` là **bắt buộc**: image Kaggle dùng timm 1.x, mà `models/models_vit.py`kế thừa `timm.models.vision_transformer.VisionTransformer` — API đổi giữa 2 major version.Không cài đè torch/torchaudio (sẽ mất bản CUDA của Kaggle).

In [ ]:
!pip install -q timm==0.9.16 einops==0.7.0 librosa==0.10.1import torch, timmprint("torch", torch.__version__, "| timm", timm.__version__,      "| GPU", torch.cuda.device_count(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 4. Xem chính xác cái gì đang được mountIn cây 3 tầng của mọi dataset, tự đánh dấu chỗ nào là *clip folders*, chỗ nào là *audio*.Cell này chỉ để **đối chiếu bằng mắt**. Nếu Cell 5 tự dò ra đúng thì không cần chép gì cả.

In [ ]:
!python tools/kaggle_setup.py --diagnose

## 5. Dựng cây symlinkĐường dẫn của bạn **đã khớp đúng branch** của dataloader (có sẵn `mfaw`/`clips_faces`,`clip_224x224`). Vấn đề nằm ở chỗ khác: loader suy ra đường dẫn `.wav` **từ đường dẫnframe bằng phép thay chuỗi**, mà frames và audio nằm ở **mount khác nhau**, và framescòn bị chia nhiều shard.Hai dataset hỏng theo hai kiểu:- **DFEW → crash** (nhánh này không có fallback).- **MAFW → hỏng im lặng**: thiếu wav thì loader thay bằng `torch.zeros(512,128)`.  Training chạy hết, có accuracy, nhưng **audio đã chết** và so sánh UOT thành vô nghĩa.Cell này dựng đúng cấu trúc loader mong đợi bằng symlink (tốn ~0 dung lượng),**không sửa một dòng nào của code baseline**.`--auto` dò theo **nội dung** chứ không theo tên: thư mục có ≥5 thư mục con chứa ảnh →*frames root*; thư mục chứa `.wav` → *audio root*; rồi lọc `mafw`/`mfaw` hoặc `dfew`.Đọc kỹ output:- `LAYOUT OK` — bắt buộc- số dòng `FRAMES` phải khớp số shard đã attach (**MAFW: 2, DFEW: 4**)- `clips w/o wav` — **phải = 0 với DFEW**; với MAFW phải nhỏ, xem cảnh báo ở trên

In [ ]:
!python tools/kaggle_setup.py --dataset {DATASET} {SRC_ARG} --out {DATA}

## 6. Trỏ annotation vào cây vừa dựng`--recount` bắt buộc: bộ preprocess của bạn gần như chắc chắn ra số frame khác bản gốc,mà loader dùng cột đó để sample index.Điều kiện đi tiếp: `missing folder: 0`, `frame count off: 0`, **`UNMATCHED path: 0`**,`labels seen` = `0..10` (MAFW) hoặc `0..6` (DFEW).

In [ ]:
!cp -r annotation annotation.bak!python tools/retarget_annotations.py --dataset {DATASET} --new-root {FRAMES_ROOT} --recount --drop-missingprint("=" * 60)!python tools/check_data.py --annotation {ANN} --n 300print("=" * 60)!python tools/check_data.py --annotation {ANN_TEST} --n 300

## 7. Checkpoint pretrainCần đúng **2 file encoder** (tên bị hardcode trong `models/Generate_Model.py`):| Cần | Là gì ||---|---|| `mae_face_pretrain_vit_base.pth` | encoder **thị giác** MAE-Face ViT-B || `audiomae_pretrained.pth` | encoder **âm thanh** AudioMAE ViT-B |Model `modelMMA` bạn đã attach chứa nhiều hơn thế, và **không phải file nào cũng dùngđể train**:| File trong mount | Thực chất | Dùng làm gì ||---|---|---|| `pretrained.pth` | encoder AudioMAE | ✅ đổi tên → `audiomae_pretrained.pth` || `mae_face_visualize_vit_base.pth` | MAE-Face **bản có decoder** | ⚠ không phải bản `pretrain`, xem dưới || `checkpoint/MAFW_224/fold*.pth` | model MMA-DFER **đã train xong** | ❌ không dùng để train — nhưng xem Cell 7b || `checkpoint/DFEW_224/fold*.pth` | như trên | ❌ / Cell 7b |`GenerateModel` nạp encoder bằng `strict=False`, nên **đưa nhầm file sẽ không báo lỗi** —nó im lặng nạp gần như không có gì rồi train từ đầu. Cell dưới chạy`tools/check_ckpt.py` để nhận diện từng file trước khi dùng.

In [ ]:
# Tự tìm mọi .pth dưới /kaggle/input rồi nhận diện từng file.# (Bỏ qua các thư mục clip nên không bị chậm vì hàng nghìn thư mục ảnh.)!python tools/check_ckpt.py --find

Đặt tên đúng như code mong đợi. `check_ckpt.py` ở trên phải xác nhận:`pretrained.pth` = **AUDIO encoder**, file vision = **VISION encoder**.Nếu `mae_face_visualize_vit_base.pth` bị báo là *"MAE with decoder"*: bản `visualize`gồm cả decoder, khác bản `pretrain` mà repo yêu cầu. Dưới `strict=False` phần encodervẫn nạp được, nhưng **hãy đọc dòng `Image checkpoint loading:` in ra ở Cell 8** — nếu`missing_keys` dài thì nó không nạp được gì đáng kể, và cần tải đúng`mae_face_pretrain_vit_base.pth` từ[MAE-Face releases](https://github.com/FuxiVirtualHuman/MAE-Face/releases).

In [ ]:
# Đường dẫn thật của Kaggle Models: /kaggle/input/models/<user>/<model>/<fw>/<variation>/<version># Lưu ý cả 2 file đều nằm trong checkpoint/, không phải ở gốc mount.MODEL_DIR = "/kaggle/input/models/tunalmt/modelmma/pytorch/default/1"CKPT      = f"{MODEL_DIR}/checkpoint"!cp {CKPT}/pretrained.pth ./audiomae_pretrained.pth!cp {CKPT}/mae_face_visualize_vit_base.pth ./mae_face_pretrain_vit_base.pth!ls -la *.pth

### 7b. (khuyên chạy) Dùng checkpoint đã train của tác giả để kiểm chứng datasetĐây là công dụng thật của thư mục `checkpoint/MAFW_224` và `checkpoint/DFEW_224`:chạy model **đã train xong của tác giả** trên dữ liệu **bạn tự preprocess**. Nếu ra sốgần với số tác giả công bố thì toàn bộ đường ống dữ liệu (frame, audio, annotation,sample rate) là đúng. Nếu ra số thấp bất thường thì có chỗ hỏng — và biết điều đó**trước khi** đốt hàng giờ GPU thì rẻ hơn nhiều.Với DFEW fold 1 @224, README dataset của bạn ghi mốc **UAR 63.63 / WAR 73.99**.Không truyền `--use-uot`: checkpoint của tác giả là kiến trúc baseline.

In [ ]:
# Chỉ chạy được sau khi Cell 5-6 đã dựng xong dữ liệu cho ĐÚNG dataset này.CKPT_EVAL = f"{CKPT}/{DATASET}_224/fold{FOLD}_224.pth"!python evaluate.py --dataset {DATASET} --fold {FOLD} --img-size 224 --checkpoint {CKPT_EVAL}

## 8. Smoke test — đừng bỏ quaTốn ~1 phút, bắt đúng các lỗi mà nếu không sẽ chỉ lộ ra sau vài giờ train.Cần thấy:- **`audio liveness ... 0/N dead`** — quan trọng nhất. Nếu `N/N dead` thì các file `.wav`  không được tìm thấy và MAFW đang chạy với audio là hằng số. **Dừng lại, đừng train.**- `SMOKE TEST PASSED`- gradient của **gate** khác 0 (gradient của `proj`/`norm` **bằng 0 là đúng** ở step đầu:  chúng nằm sau `tanh(gate)=0`)- `max|baseline - uot| ... OK` — model khởi đầu trùng khít baseline- `peak GPU memory` — cho biết `BATCH` nào vừa VRAM

In [ ]:
!python tools/smoke_test.py --dataset {DATASET} --use-uot --batch-size 2

## 9. Train — baselineChạy baseline **trước** để có mốc so sánh. Cùng seed (hardcode `seed=1`), cùng fold,cùng số epoch với run UOT.`--folds` là cờ thêm vào `main.py`: mặc định script gốc chạy cả 5 fold liên tiếp trongmột process, không thể xong trong giới hạn phiên của Kaggle.

In [ ]:
!python main.py --dataset {DATASET} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --exper-name KAGGLE_BASE

## 10. Train — UOTKhác đúng một thứ: `--use-uot`. Ablation quan trọng nhất là `--uot-tau 1e6`(≈ balanced OT) — nếu nó ngang bằng `--uot-tau 1.0` thì tính "unbalanced" không đónggóp gì. Chi tiết các nút vặn: `UOT_CODE_WALKTHROUGH.md` §3.

In [ ]:
!python main.py --dataset {DATASET} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --use-uot --uot-eps 0.05 --uot-tau 1.0 --uot-iters 10 \  --exper-name KAGGLE_UOT_tau1.0

## 11. Kết quả

In [ ]:
import glob, refor log in sorted(glob.glob("log/*/log.txt")):    txt = open(log).read()    accs = re.findall(r"Current Accuracy: ([\d.]+)", txt)    uar  = re.findall(r"UAR: ([\d.]+)", txt)    war  = re.findall(r"WAR: ([\d.]+)", txt)    ep   = re.findall(r"An epoch time: ([\d.]+)", txt)    name = log.split("/")[1]    print(f"{name}")    print(f"   val acc mỗi epoch : {accs}")    print(f"   UAR / WAR         : {uar} / {war}")    if ep:        m = sum(float(e) for e in ep) / len(ep) / 60        print(f"   epoch trung bình  : {m:.1f} phút  ->  25 epoch x 5 fold ~ {m*25*5/60:.1f} giờ")    print()

## Ghi chú- Kết quả nằm ở `/kaggle/working/DEFR-UOT/log/` → được giữ khi **Save Version** (quota 20 GB).- Phiên tương tác bị ngắt khi idle. Chạy dài thì dùng **Save Version → Save & Run All (Commit)**.- `/kaggle/temp` bị xoá sau mỗi phiên → chạy lại Cell 5 mỗi lần mở notebook (vài giây).- Dùng số `epoch trung bình` ở Cell 11 để ước lượng ngân sách thật trên server.